# Dataset preparation with bundled example data

Run all cells in order with the Python 3 kernel from the environment in README.md.
The default example needs no downloads or path edits. It contains three real frames
and their original YOLO labels. This is a preparation demonstration, not model training
or raw sonar processing. Inputs are never rewritten.


In [ ]:
from pathlib import Path
import importlib.util

# Jupyter normally starts the kernel in this notebook's directory.
# Also accept a working directory anywhere beneath the repository root.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "prepare_dataset.py").is_file()
                     and (p / "examples" / "input_data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from its repository directory (see README.md).")
spec = importlib.util.spec_from_file_location("prepare_dataset", PROJECT_ROOT / "prepare_dataset.py")
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)


## Configuration

For the quick test, leave this cell unchanged. For the full dataset, change only
`DATA_ROOT` to the directory containing the extracted `images/` and `labels/`
folders (or original per-line folders). `OUTPUT_ROOT = None` creates a new run
folder beneath `example_output/` on every execution. A custom output path must
not exist and must not overlap the input directory.


In [ ]:
DATA_ROOT = PROJECT_ROOT / "examples" / "input_data"
OUTPUT_ROOT = None


## Validate and prepare

The shared script checks matching image names, YOLO fields, normalized coordinates
and class ID 0 before creating any output. Unexpected IDs,
raise an error; annotations are never remapped. Each image and label is copied
byte for byte. Frames without bounding-box annotations need no label file.


In [ ]:
metadata, output_dir = workflow.prepare_dataset(DATA_ROOT, OUTPUT_ROOT)
metadata


## Summary and output check

The bundled example should yield 3 survey-line IDs, 3 frames, 3 annotated frames,
and the bounding-box count recorded in `examples/README.md`.
The full dataset has 67,681 frames, 26,358 annotated frames and 188,684 boxes;
the 41,323 remaining frames are unlabelled/background frames.


In [ ]:
assert (output_dir / "metadata.csv").is_file()
assert (output_dir / "classes.txt").read_text().strip() == "vegetation_like_target"
print(metadata[["total_frames", "annotated_frames", "total_bboxes"]].sum())
print(f"Generated files: {sum(p.is_file() for p in output_dir.rglob('*'))}")
print(f"Metadata: {output_dir / 'metadata.csv'}")


## Use in machine learning

No splits are generated. Random image-level splitting can lead to data leakage
because temporally adjacent water-column frames may be highly similar or associated
with the same sonar ping. Prefer survey-line-disjoint, temporally blocked, or
spatially separated splits, and check for residual overlap. Missing annotations
do not establish biological absence.
